# 01. EDA

Stage 1.2 of `docs/codeagents/research_plan.md` (§2.3). All statistics are computed on the **train** split only; val/test appear once, to measure how well a train-built vocabulary covers their Listen+ targets.

Logic lives in `recsys_rl.eda`. Run `00_colab_setup.ipynb` once to download the data to Drive; the first cell below installs the package in this notebook's runtime.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import numpy as np
import polars as pl

from recsys_rl import data, eda
from recsys_rl.config import load_config

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(160)

cfg = load_config("configs/base.yaml")
events = data.scan_events(cfg)
bounds = data.SplitBounds.from_events(events, cfg.data)
splits = data.temporal_split(events, bounds)
train = splits["train"]
listens = train.filter(data.is_listen())
print(bounds)

## Played ratio and track length

In [ ]:
display(eda.ratio_summary(listens))
eda.ratio_histogram(listens, edges=[0, 5, 10, 25, 49, 75, 99, 100])

In [ ]:
print(eda.length_quantiles(listens, [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
by_length = eda.ratio_by_length(listens, edges=[60, 120, 180, 240, 300, 420, 600])
eda.plot_ratio_by_length(by_length)
by_length

## First vs repeated listens

In [ ]:
eda.first_vs_repeat(listens)

## What does `is_organic` mean?

The HF card is ambiguous. Library/search listens should be mostly repeats, and unlikes/undislikes happen mostly while browsing one's own library.

In [ ]:
eda.organic_profile(train)

## Sessions and daily activity

Session = run of listens without a pause > `session_gap_minutes` (30).

In [ ]:
sess = eda.sessions(listens, cfg.data.session_gap_minutes)
qs = [0.25, 0.5, 0.75, 0.9, 0.99]
display(
    sess.select(
        [pl.col("size").quantile(q).alias(f"size_q{q}") for q in qs]
        + [pl.col("duration_min").quantile(q).alias(f"dur_min_q{q}") for q in qs]
        + [pl.col("pause_h").quantile(q).alias(f"pause_h_q{q}") for q in qs]
    )
)
print(len(sess), "sessions;", eda.daily_activity(listens))
eda.plot_hist(sess["size"].to_numpy(), np.logspace(0, 3.5, 40), "session size, listens", logx=True);

## Popularity tail and vocabulary size

In [ ]:
ranks = eda.item_ranks(train)
sizes = [10_000, 50_000, 100_000, 200_000, 300_000, 500_000]
coverage = eda.vocab_coverage(ranks, splits, sizes, cfg.data.listen_plus_threshold)
print(len(ranks), "items listened in train")
eda.plot_vocab_coverage(coverage)
coverage

## Findings (train split, 46.0M listens)

Numbers are from the run on Yambda-50M; re-running the notebook reproduces them.

1. **Played ratio is bimodal.** 53% of listens are full plays (ratio = 100), 24% are skips (≤ 5%, of which 7.5 pp are exactly 0), only ~18% lie in between; 0.5% exceed 100 (rewinds). Listen+ (≥ 50%) = 63%. Bins for feedback tokens should follow this shape: `0, (0,5], (5,10], (10,25], (25,50), [50,75], (75,100), 100, >100` rather than uniform deciles.
2. **Ratio falls with track length; listened seconds grow.** Mean ratio drops monotonically from 73% (≤ 60 s) to 48% (> 10 min) and Listen+ from 71% to 47%, while mean listened time rises from 32 s to 582 s. A ratio reward therefore favours short tracks and a listen-time reward favours long ones: this is the main reward-hacking axis to track (mean recommended track length in every RL run; R-variants with length normalisation). Lengths: median 200 s, 5–95% range 125–310 s.
3. **Repeats are listened to more fully** than first listens (+4–9 pp mean ratio at a fixed `is_organic`). A ratio-based reward rewards recommending already-heard tracks, so the repeat rate must be reported for every policy (and the repeat-filter evaluation mode kept separate).
4. **`is_organic = 1` means organic (library/search), 0 means recommendations**, i.e. the table on the HF card is right. Evidence: 81% of `is_organic = 1` listens are repeats vs 62%, and unlikes are 17× more frequent (12.4 vs 0.7 per 1k listens), which fits browsing one's own collection. Counter-intuitively, organic listens have *more* skips (31% vs 22% with ratio < 10%) and a lower mean ratio (60% vs 70%) — consistent with shuffling through a library. The two sources are balanced (52% / 48% of listens).
5. **Sessions** (30-min gap): 2.43M sessions, median 10 listens / 23 min, 90th percentile 44 listens / 102 min; median pause between sessions 6.2 h. Users are very regular: 74% of user-days are followed by an active next day, median 91 active days per user. A window of 256 listens spans ~13 sessions on average (mean session: 19 listens), so long-term (session/return) signals are learnable from context.
6. **Long tail and vocabulary.** 873k items are listened in train. Top-200k items (min 15 listens) cover 95.4% of train listens but only 91.2% of test Listen+ targets (93.8% of val); 300k (min 6) would give 93.7% on test. ~1% of test targets are never seen in train. Coverage on test is the hard ceiling for Recall, so `max_vocab_size = 200_000` costs ≤ 2.5 pp of reachable targets vs 300k at 2/3 of the embedding size — keep 200k unless the baseline turns out to be capacity-bound.